# Notebook 06 — Clustering & Statistical Inference
**Contributor:** SKYLearn-Innovation Tutor (university undergraduate)  
**Reviewed by:** Dingaan Mahlatse Machethe  
**Tier:** TUTOR (intermediate-advanced)  

Two parts:
1. **Unsupervised learning** — K-Means clustering of provinces by performance profile,
   with silhouette analysis to choose k and PCA for 2D visualisation
2. **Statistical inference** — formal hypothesis tests (t-test, ANOVA, correlation, regression)

In [ ]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from preprocess import load_clean
from cluster import build_province_profile, find_optimal_k, cluster_provinces
from stats_tests import run_all_tests

sns.set_theme(style='whitegrid')
df = load_clean()
print('Dataset:', df.shape)

## Part 1 — K-Means Clustering
### 1.1 Build per-province feature profile

In [ ]:
profile = build_province_profile(df)
profile.round(2)

### 1.2 Silhouette analysis — choose optimal k

In [ ]:
sil = find_optimal_k(profile)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(sil['k'], sil['silhouette'], 'o-', color='#1F4E79', linewidth=2)
ax1.set_title('Silhouette Score vs k', fontweight='bold'); ax1.set_xlabel('k'); ax1.set_ylabel('Silhouette')
ax2.plot(sil['k'], sil['inertia'], 's-', color='#117A65', linewidth=2)
ax2.set_title('Elbow (Inertia) vs k', fontweight='bold'); ax2.set_xlabel('k'); ax2.set_ylabel('Inertia')
plt.tight_layout(); plt.show()
sil

### 1.3 Cluster provinces (k=3) and project with PCA

In [ ]:
result = cluster_provinces(profile, k=3)
labelled = result['labelled']
print(f"Silhouette: {result['silhouette']}  |  PCA variance explained: {sum(result['explained_variance'])*100:.0f}%")

fig, ax = plt.subplots(figsize=(10, 7))
palette = {'High Performing': '#117A65', 'Mid Performing': '#1F4E79', 'Needs Support': '#E74C3C'}
for tier, grp in labelled.groupby('cluster_name'):
    ax.scatter(grp['pca_x'], grp['pca_y'], s=200, label=tier, color=palette.get(tier, '#888'))
    for _, row in grp.iterrows():
        ax.annotate(row['province'], (row['pca_x'], row['pca_y']), fontsize=8,
                    xytext=(5, 5), textcoords='offset points')
ax.set_title('Province Clusters (PCA projection)', fontweight='bold')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
labelled[['province', 'cluster_name', 'stem_pass_rate', 'nonstem_pass_rate',
          'absentee_rate', 'pass_rate_volatility']].round(2).sort_values('stem_pass_rate', ascending=False)

## Part 2 — Statistical Hypothesis Testing (α = 0.05)

In [ ]:
results = run_all_tests(df)
for name, res in results.items():
    print(f"\n{'='*55}\n{name}\n{'='*55}")
    for k, v in res.items():
        print(f"  {k:20s}: {v}")

In [ ]:
# Visualise the STEM vs Non-STEM distribution (the t-test/Mann-Whitney comparison)
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df, x='subject_type', y='pass_rate', palette=['#1F4E79', '#117A65'], ax=ax)
ax.set_title('Pass Rate Distribution — STEM vs Non-STEM', fontweight='bold')
ax.set_xlabel(''); ax.set_ylabel('Pass Rate (%)')
plt.tight_layout(); plt.show()

## Key Findings
1. Provinces cluster into **3 clear tiers**: High Performing, Mid Performing, Needs Support
2. The **t-test/Mann-Whitney** confirms STEM pass rates are significantly lower than Non-STEM (p < 0.001)
3. **ANOVA** confirms provinces differ significantly in mean pass rate (p < 0.001)
4. These statistically-validated groupings let SKYLearn-Innovation target resources where they matter most